In [ ]:
import Pkg

Pkg.activate(".")
Pkg.instantiate()

In [ ]:
using NumericalEarth
using Oceananigans
using Oceananigans.Units
using Oceananigans.Grids: node
using Oceananigans.OutputReaders: Clamp
using Oceananigans.TurbulenceClosures: VerticalScalarDiffusivity, VerticallyImplicitTimeDiscretization
using Dates
using Printf
using Statistics
using CUDA

In [ ]:
# determining run version/params - :dye_test or :spinup
run_mode = :dye_test

use_rivers = true
resolution_tag = "trial4_12_ecco1day_restoring_kz1e_3_original_rivers_0p1deg"

# trial 4.12: trial 4.11 with stronger one-day ecco salinity restoring.

if run_mode == :dye_test
    run_days = 12
elseif run_mode == :spinup
    run_days = 360
else
    error("need right mode")
end

if use_rivers
    run_name = "rivers_on"
else
    run_name = "rivers_off"
end

if run_mode == :dye_test
    output_tag = "$(run_name)_dye_test_$(resolution_tag)"
else
    output_tag = "$(run_name)_spinup"
end

surface_filename = "amazon_$(output_tag)_surface_fields"
free_surface_filename = "amazon_$(output_tag)_free_surface"
dye_3d_filename = "amazon_$(output_tag)_dye_3d"
salinity_3d_filename = "amazon_$(output_tag)_salinity_3d"

checkpoint_filename = "amazon_$(run_name)_spinup_checkpoint.jld2"

In [ ]:

arch = GPU()

# grid definitions -- amazon river mouth / plume region
long_west = -58.5
long_east = -41.5
lat_south = -7.8
lat_north = 9.2

# trial 4.12 retains the baseline 0.1-degree horizontal resolution.
# keep the 30 m minimum depth so river mixing has an active cell below.
Nx = 170
Ny = 170
Nz = 20

long_river = -49.5
lat_river  = 0.16

depth = 4000meters # decreased depth to 4k for this test 

z = ExponentialDiscretization(Nz, -depth, 0; scale = depth/4, mutable = false)
underlying_grid = LatitudeLongitudeGrid(
    arch;
    size = (Nx, Ny, Nz),
    halo = (5, 5, 4),
    longitude = (long_west, long_east),
    latitude = (lat_south, lat_north),
    z,
    topology = (Bounded, Bounded, Bounded)
)

# use at least 30 m so river-mouth columns can mix below the surface cell
bottom_height = regrid_bathymetry(underlying_grid;
                                  minimum_depth = 30,
                                  interpolation_passes = 10, 
                                  major_basins = 1) # only one major basin here -- atlantic 

grid = ImmersedBoundaryGrid(underlying_grid, GridFittedBottom(bottom_height);
                            active_cells_map=true)


                        

In [ ]:

# trial 4.12 focuses on river salinity, so redi is left out.
vertical_mixing = NumericalEarth.Oceans.default_ocean_closure()

# add extra vertical tracer mixing near the river mouth and above 30 meters
const river_mixing_longitude = -49.35
const river_mixing_latitude = -0.15
const river_mixing_kz = 1e-3
const river_mixing_depth = 30meters
const river_mixing_half_width = 0.75

@inline function river_mouth_kz(longitude, latitude, z, time)
    inside_longitude = abs(longitude - river_mixing_longitude) <= river_mixing_half_width
    inside_latitude = abs(latitude - river_mixing_latitude) <= river_mixing_half_width
    inside_depth = z >= -river_mixing_depth

    if inside_longitude && inside_latitude && inside_depth
        return river_mixing_kz
    else
        return 0.0
    end
end

river_mixing = VerticalScalarDiffusivity(
    VerticallyImplicitTimeDiscretization();
    κ = (T = river_mouth_kz,
         S = river_mouth_kz,
         dye = river_mouth_kz,
         e = 0.0)
)

closure = (vertical_mixing, river_mixing)

In [ ]:
# ecco setup must come before the restoring object is constructed
ENV["ECCO_USERNAME"] = "meggoeggo"
ENV["ECCO_WEBDAV_PASSWORD"] = "3a0bqyAsTojLw4ZtpKaC"
date = DateTime(1993, 1, 1)

free_surface = SplitExplicitFreeSurface(grid; substeps = 70)
momentum_advection = WENOVectorInvariant(order = 5)
tracer_advection = WENO(order = 5)

# create zero-gradient (neumann) boundary conditions for dye 
# "flow out" part of the model 
dye_bcs = FieldBoundaryConditions(
    west   = GradientBoundaryCondition(0),
    east   = GradientBoundaryCondition(0),
    south  = GradientBoundaryCondition(0),
    north  = GradientBoundaryCondition(0),
    top    = FluxBoundaryCondition(0),
    bottom = FluxBoundaryCondition(0)
)

# compile tracer boundary conditions for the model
model_bcs = (
    dye = dye_bcs,
)

# updated sponge layer logic!
# summary: inital sponge was too large + errored. kept on throwing invalidireerror. oceanangians converts the sponge relaxation into a continuous form, which isn't compatible with the gpu, hence the error
# made a manual function creating the gaussian mask for the northern bounday, but then applied the relaxation in discrete form 

# store as gpu-accessible constant
const north_sponge_mask = GaussianMask{:y}(
    center = 9.2,
    width = 0.25
)

# store as a gpu-accessible constant
const north_sponge_rate = 1 / 5days

# apply gaussian dye relaxation in discrete form
@inline function north_dye_sponge(i, j, k, grid, clock, model_fields)
    # get the coordinates of dye cell
    x, y, z = node(i, j, k, grid, Center(), Center(), Center())

    # evaluate gaussian mask from oceaningans at this cell
    mask = north_sponge_mask(x, y, z)

    # read the local dye concentration + relax towards 0 
    dye = @inbounds model_fields.dye[i, j, k]
    return -north_sponge_rate * mask * dye

end

# use the discrete form to avoid the bug 
dye_sponge = Forcing(
    north_dye_sponge;
    discrete_form = true
)

# restore salinity toward the january 1993 ecco field over one day
salinity_metadata = Metadata(:salinity; dataset = ECCO4Monthly(), dates = [date])
salinity_restoring = DatasetRestoring(
    salinity_metadata,
    grid;
    rate = 1 / 1days,
    time_indices_in_memory = 1,
    time_indexing = Clamp()
)

# keep the dye sponge and add the salinity restoring as model forcings
model_forcing = (
    dye = dye_sponge,
    S = salinity_restoring,
)
                        
# dd dye with the original numericalearth momentum, tracer, and closure settings.
ocean = ocean_simulation(grid;
                         momentum_advection, tracer_advection, free_surface,
                         closure = closure,
                         tracers = (:T, :S, :dye),
                         boundary_conditions = model_bcs,
                         forcing = model_forcing
                         )

# gaussian dye patch centered on the river mouth across multiple cells
# dye is initially deposited only in the upper 10 meters
@inline function dye_initial_condition(x, y, z)
    horizontal_width = 0.5
    horizontal_blob = exp(-((x - long_river)^2 + (y - lat_river)^2) / horizontal_width^2)

    if z > -10
        return horizontal_blob
    else
        return 0.0
    end
end

# print the model structure so we can check that dye is listed as a tracer and that the boundary conditions were set to 0 gradient
@info "We've built an ocean simulation with model:"
@show ocean.model
@show ocean.model.tracers.dye.boundary_conditions



In [ ]:
# ask for ecco credentials at runtime so they are not saved in the notebook



ecco_variables = (:temperature, :salinity)
ecco_set = MetadataSet(ecco_variables; dataset = ECCO4Monthly(), date)

# we aren't initializing the tracer just yet (first spinning-up the model for a year)  
# print out all of the tracers 
set!(ocean.model, ecco_set)
@show keys(ocean.model.tracers)

In [ ]:

land = JRA55PrescribedLand(arch)
atmosphere = JRA55PrescribedAtmosphere(arch)
ocean_surface = SurfaceRadiationProperties(albedo = LatitudeDependentAlbedo())
radiation = JRA55PrescribedRadiation(arch; ocean_surface)

In [ ]:
# use numericalearth's original jra55 river forcing without custom spreading

In [ ]:
# trial 4.12 requires the river freshwater forcing
@assert use_rivers "trial 4.12 requires use_rivers = true"
land_component = land

# build the regional ocean model
coupled_model = EarthSystemModel(
    ;
    ocean,
    atmosphere,
    land = land_component,
    radiation
)

# check that the extra mixing turns off below 30 meters
@assert river_mouth_kz(river_mixing_longitude, river_mixing_latitude, -10, 0) == river_mixing_kz
@assert river_mouth_kz(river_mixing_longitude, river_mixing_latitude, -40, 0) == 0.0
@info "trial 4.12 river mixing check" river_mixing_longitude river_mixing_latitude river_mixing_kz river_mixing_depth river_mixing_half_width restoring_rate=salinity_restoring.rate
@assert salinity_restoring.rate == 1 / 1days "trial 4.12 expected one-day restoring"
trial4_12_preflight_passed = true

# timestep used for both the 12-day test and 360-day spin-up
time_step = 3minutes

simulation = Simulation(
    coupled_model;
    Δt = time_step,
    stop_time = run_days * days
)

# print advective cfl in log 
advective_cfl = AdvectiveCFL(time_step)


In [ ]:
# no custom river-spreading diagnostics are needed in trial 4.12

In [ ]:
@info "trial 4.12 uses original rivers plus stronger direct ecco salinity restoring"


### trial 4.3 result - river forcing location check

this diagnostic used the saved freshwater flux arrays and did not rerun the simulation. within two degrees of the assumed amazon mouth, 440 ocean cells had positive spread freshwater flux. the trial 4.2 mixing box contained 12 ocean cells, overlapped 10 forced cells, and captured 1.3940e7 of the local 1.0263e8 freshwater transport, or 13.58 percent.

the strongest forced cell was at 49.35 degrees west and 0.15 degrees south with a flux of 0.0838237. the nine strongest forced cells were all outside the trial 4.2 mixing box; only the tenth-ranked cell was inside. trial 4.2 therefore did not place its heightened diffusivity over the main freshwater-input cluster.

the next test should keep the 3 by 3 freshwater spreading and upper-30-meter diffusivity, but center the mixing region on the measured freshwater cluster instead of the original dye-release coordinates.

In [ ]:
wall_time = Ref(time_ns())

function progress(sim)

    ocean = sim.model.ocean

    u, v, w = ocean.model.velocities

    T = ocean.model.tracers.T
    S = ocean.model.tracers.S
    dye = ocean.model.tracers.dye
    e = ocean.model.tracers.e

    Tmin, Tmax = minimum(T), maximum(T)
    Smin, Smax = minimum(S), maximum(S)
    dyemin, dyemax = minimum(dye), maximum(dye)
    emin, emax = minimum(e), maximum(e)
    cfl = advective_cfl(ocean.model)

    umax = (
        maximum(abs, u),
        maximum(abs, v),
        maximum(abs, w)
    )

    step_time = 1e-9 * (time_ns() - wall_time[])

    msg1 = @sprintf(
        "time: %s, iter: %d",
        prettytime(sim),
        iteration(sim)
    )

    msg2 = @sprintf(
        ", max|u|: (%.3e, %.3e, %.3e) m s⁻¹",
        umax[1],
        umax[2],
        umax[3]
    )

    msg3 = @sprintf(
        "\nT:    min = %.6e, max = %.6e °C",
        Tmin,
        Tmax
    )

    msg4 = @sprintf(
        "\nS:    min = %.6e, max = %.6e",
        Smin,
        Smax
    )

    msg5 = @sprintf(
        "\ndye:  min = %.6e, max = %.6e",
        dyemin,
        dyemax
    )

    msg6 = @sprintf(
        "\ne:    min = %.6e, max = %.6e m² s⁻²",
        emin,
        emax
    )

    msg7 = @sprintf(
        "\nadvective CFL = %.6e",
        cfl
    )

    msg9 = @sprintf(
        "\nwall time since last report: %s\n",
        prettytime(step_time)
    )

    @info msg1 * msg2 * msg3 * msg4 * msg5 * msg6 * msg7 * msg9

    tracer_extrema = (
        Tmin, Tmax,
        Smin, Smax,
        dyemin, dyemax,
        emin, emax
    )

    if dyemin < 0
        warning_message = @sprintf(
            "NEGATIVE DYE DETECTED: min(dye) = %.6e",
            dyemin
        )
        @warn warning_message
    end

    if dyemax > 1
        warning_message = @sprintf(
            "OVERSHOOT DYE DETECTED: max(dye) = %.6e",
            dyemax
        )
        @warn warning_message
    end

    if Smin < 0
        warning_message = @sprintf(
            "NEGATIVE SALINITY DETECTED: min(S) = %.6e",
            Smin
        )
        @warn warning_message
    end

    wall_time[] = time_ns()

    return nothing
end

In [ ]:
# print the progress report every simulated day
add_callback!(simulation, progress, TimeInterval(1days))

In [ ]:
# collect surface tracers + velocities
ocean_outputs = merge(
    ocean.model.tracers,
    ocean.model.velocities
)

# collect full 3d salinity
salinity_output = (
    S = ocean.model.tracers.S,
)

# grab the free-surface displacement
free_surface = ocean.model.free_surface.displacement

# save daily surface tracers + velocities in both modes
ocean.output_writers[:surface] = JLD2Writer(
    ocean.model,
    ocean_outputs;
    schedule = TimeInterval(1days),
    filename = surface_filename,
    indices = (:, :, grid.Nz),
    overwrite_existing = true
)

# save daily sea-surface height in both modes
ocean.output_writers[:free_surface] = JLD2Writer(
    ocean.model,
    (; η = free_surface);
    schedule = TimeInterval(1days),
    filename = free_surface_filename,
    overwrite_existing = true
)

if run_mode == :dye_test
    dye_output = (
        dye = ocean.model.tracers.dye,
    )

    # save full 3d dye daily for the 5-day test
    ocean.output_writers[:dye_3d] = JLD2Writer(
        ocean.model,
        dye_output;
        schedule = TimeInterval(1days),
        filename = dye_3d_filename,
        overwrite_existing = true
    )

    # save full 3d salinity daily for the short test
    ocean.output_writers[:salinity_3d] = JLD2Writer(
        ocean.model,
        salinity_output;
        schedule = TimeInterval(1days),
        filename = salinity_3d_filename,
        overwrite_existing = true
    )

elseif run_mode == :spinup
    # do not save 3d dye because it remains zero during spin-up

    # save full 3d salinity every 30 days during spin-up
    ocean.output_writers[:salinity_3d] = JLD2Writer(
        ocean.model,
        salinity_output;
        schedule = TimeInterval(30days),
        filename = salinity_3d_filename,
        overwrite_existing = true
    )

    # save a restart checkpoint every 30 days
    simulation.output_writers[:checkpointer] = Checkpointer(
        simulation.model;
        schedule = TimeInterval(30days),
        prefix = "amazon_$(run_name)_spinup_checkpoint",
        cleanup = true,
        overwrite_existing = true
    )
end

In [ ]:
# simple check that the added river closure is attached
model_closures = ocean.model.closure
@assert length(model_closures) == 2 "trial 4.12 expected CATKE and river mixing"
@assert model_closures[2] isa VerticalScalarDiffusivity "the river mixing closure is missing from the model"
trial4_12_closure_check_passed = true


### trial 4.12 - stronger direct ecco salinity restoring

this trial keeps the trial 4.11 setup but shortens the ecco salinity-restoring timescale from ten days to one day.

all river forcing, mixing, dye, grid, and timestep settings remain the same as trial 4.11.


In [ ]:
# stop before the long run if the trial 4.12 setup was not rebuilt and checked
if !(@isdefined trial4_12_preflight_passed)
    error("run the trial 4.12 restoring check before this cell")
end
if !(@isdefined trial4_12_closure_check_passed)
    error("run the trial 4.12 closure check before this cell")
end
@assert trial4_12_preflight_passed "the trial 4.12 preflight did not pass"
@assert trial4_12_closure_check_passed "the trial 4.12 closure check did not pass"
@assert resolution_tag == "trial4_12_ecco1day_restoring_kz1e_3_original_rivers_0p1deg"
@assert river_mixing_longitude == -49.35
@assert river_mixing_latitude == -0.15
@assert river_mixing_half_width == 0.75
@assert river_mixing_depth == 30meters
@assert river_mixing_kz == 1e-3
@assert iteration(simulation) == 0 "build a new simulation before starting trial 4.12"
@info "trial 4.12 pre-run check passed" river_mixing_longitude river_mixing_latitude river_mixing_half_width river_mixing_depth river_mixing_kz restoring_rate=salinity_restoring.rate

if run_mode == :dye_test
    # deposit dye immediately for the 12-day test
    set!(ocean.model, dye = dye_initial_condition)

    @show minimum(ocean.model.tracers.dye)
    @show maximum(ocean.model.tracers.dye)

    @info "initial values right after dye is deposited:"
    progress(simulation)

elseif run_mode == :spinup
    # begin the 360-day spin-up with no dye
    set!(ocean.model, dye = 0.0)

    @info "spinup has 0 dye."
    @show minimum(ocean.model.tracers.dye)
    @show maximum(ocean.model.tracers.dye)
end
run!(simulation)

### trial 4.2 result - local upper-30-meter river mixing

the run completed all 12 simulated days in 51.3 minutes and wrote 13 daily records from day 0 through day 12. the model printout confirmed that the extra vertically implicit tracer diffusivity was included with κz = 0.1 m²/s for temperature, salinity, and dye. the 3 by 3 river spreading still conserved freshwater transport with a relative error of 1.35e-16.

this version did not improve the salinity failure. minimum salinity was 0.03020 on day 8, became negative on day 9 at -1.38576, and ended at -2.60432 on day 12. these values match trial 4.1 to the reported precision. the dye undershoot and overshoot also followed the same pattern.

trial 4.2 therefore does not show that local vertical mixing is ineffective. instead, the unchanged result suggests that the fixed longitude-latitude mixing box may not overlap the active ocean cells receiving the regridded river flux. before changing κz, the next test should print the coordinates of the nonzero spread-flux cells and build the mixing mask from those cells.

### trial 4.1 result - 3 by 3 river spreading

the run completed all 12 simulated days and wrote 13 daily records from day 0 through day 12. the spreading conserved active-ocean freshwater transport: the raw transport was 1.1018806e8, the spread transport was 1.1018806e8, and the relative error was 1.35e-16. the maximum local freshwater flux decreased from 0.12167 to 0.08382.

the 3 by 3 spreading was not enough to keep salinity physical. minimum salinity fell from 31.02885 initially to 20.40982 on day 1, reached 0.03020 on day 8, became negative on day 9 at -1.38576, and ended at -2.60432 on day 12. the advective cfl remained below 0.018, so this result is not explained by a large advective timestep.

ordinary redi again produced the known dye undershoot and overshoot. dye reached a minimum of -2.714e-4 on day 6 and a maximum of 1.02731 on day 8. this is separate from the salinity result. trial 4.1 therefore rejects 3 by 3 horizontal spreading as a complete fix; the next test should keep the spreading and add stronger near-surface vertical mixing through about 30 meters.

### Trial 4.0 result - ordinary Redi, 0.1 degree, rivers off

The run completed all 12 simulated days and wrote 13 daily records (days 0 through 12). Minimum salinity stayed positive and close to its initial value: 31.02885 initially and 30.84038 on day 12. This contrasts with the rivers-on 0.1-degree baseline, where salinity became negative. The result strongly supports concentrated river freshwater forcing as the cause of the fine-grid salinity collapse.

Ordinary Redi reproduced the previously observed dye behavior: dye first became negative on day 1 (-1.21e-5), reached its most negative daily value on day 6 (-2.69e-4), and was -1.60e-4 on day 12. Dye exceeded one beginning on day 7 and reached 1.0171 on day 12. This is expected for this salinity-control trial and remains separate from the river-salinity result.

In [ ]:

# add the file extension used by fieldtimeseries
surface_filepath = surface_filename * ".jld2"
free_surface_filepath = free_surface_filename * ".jld2"
dye_3d_filepath = dye_3d_filename * ".jld2"
salinity_3d_filepath = salinity_3d_filename * ".jld2"

# load daily surface fields
uo = FieldTimeSeries(surface_filepath, "u"; backend = OnDisk())
vo = FieldTimeSeries(surface_filepath, "v"; backend = OnDisk())
To = FieldTimeSeries(surface_filepath, "T"; backend = OnDisk())
So = FieldTimeSeries(surface_filepath, "S"; backend = OnDisk())
eo = FieldTimeSeries(surface_filepath, "e"; backend = OnDisk())
dye = FieldTimeSeries(surface_filepath, "dye"; backend = OnDisk())
ηo = FieldTimeSeries(free_surface_filepath, "η"; backend = OnDisk())

# load full 3d dye saved daily
dye_3d = FieldTimeSeries(dye_3d_filepath, "dye"; backend = OnDisk())
# load full 3d salinity saved every 5 days
S_3d = FieldTimeSeries(salinity_3d_filepath, "S"; backend = OnDisk())

# check the output times + dimensions
@show length(So.times)
@show So.times ./ days
@show length(dye_3d.times)
@show dye_3d.times ./ days
@show length(S_3d.times)
@show S_3d.times ./ days
@show size(interior(dye_3d[1]))
@show size(interior(S_3d[1]))

In [ ]:

using CairoMakie

times = To.times
Nt = minimum((length(uo.times), length(vo.times), length(To.times), length(So.times), length(eo.times), length(dye.times), length(ηo.times)))
n = Observable(1)

In [ ]:
# land mask from bathymetry
land_full = interior(To.grid.immersed_boundary.bottom_height) .≥ 0

Toₙ = @lift begin
    Tₙ = Array(interior(To[$n]))

    land = falses(size(Tₙ))
    i = min(size(Tₙ, 1), size(land_full, 1))
    j = min(size(Tₙ, 2), size(land_full, 2))
    k = min(size(Tₙ, 3), size(land_full, 3))
    land[1:i, 1:j, 1:k] .= land_full[1:i, 1:j, 1:k]

    Tₙ[land] .= NaN
    view(Tₙ, :, :, 1)
end

eoₙ = @lift begin
    eₙ = Array(interior(eo[$n]))

    land = falses(size(eₙ))
    i = min(size(eₙ, 1), size(land_full, 1))
    j = min(size(eₙ, 2), size(land_full, 2))
    k = min(size(eₙ, 3), size(land_full, 3))
    land[1:i, 1:j, 1:k] .= land_full[1:i, 1:j, 1:k]

    eₙ[land] .= NaN
    view(eₙ, :, :, 1)
end

ηoₙ = @lift begin
    ηₙ = Array(interior(ηo[$n]))

    land = falses(size(ηₙ))
    i = min(size(ηₙ, 1), size(land_full, 1))
    j = min(size(ηₙ, 2), size(land_full, 2))
    k = min(size(ηₙ, 3), size(land_full, 3))
    land[1:i, 1:j, 1:k] .= land_full[1:i, 1:j, 1:k]

    ηₙ[land] .= NaN
    view(ηₙ, :, :, 1)
end

uoₙ = Field{Face, Center, Nothing}(uo.grid)
voₙ = Field{Center, Face, Nothing}(vo.grid)
so = Field(sqrt(uoₙ^2 + voₙ^2))

soₙ = @lift begin
    parent(uoₙ) .= parent(uo[$n])
    parent(voₙ) .= parent(vo[$n])
    compute!(so)

    sₙ = Array(interior(so))

    land = falses(size(sₙ))
    i = min(size(sₙ, 1), size(land_full, 1))
    j = min(size(sₙ, 2), size(land_full, 2))
    k = min(size(sₙ, 3), size(land_full, 3))
    land[1:i, 1:j, 1:k] .= land_full[1:i, 1:j, 1:k]

    sₙ[land] .= NaN
    view(sₙ, :, :, 1)
end

dyeₙ = @lift begin
    d = Array(interior(dye[$n]))

    land = falses(size(d))
    i = min(size(d, 1), size(land_full, 1))
    j = min(size(d, 2), size(land_full, 2))
    k = min(size(d, 3), size(land_full, 3))
    land[1:i, 1:j, 1:k] .= land_full[1:i, 1:j, 1:k]

    d[land] .= NaN
    d_log = log10.(max.(d, 0) .+ 1e-16)

    view(d_log, :, :, 1)
end

Soₙ = @lift begin
    Sₙ = Array(interior(So[$n]))

    land = falses(size(Sₙ))
    i = min(size(Sₙ, 1), size(land_full, 1))
    j = min(size(Sₙ, 2), size(land_full, 2))
    k = min(size(Sₙ, 3), size(land_full, 3))
    land[1:i, 1:j, 1:k] .= land_full[1:i, 1:j, 1:k]

    Sₙ[land] .= NaN
    view(Sₙ, :, :, 1)
end

In [ ]:
fig = Figure(size = (1200, 1600))

title = @lift string("amazon river snapshot ", prettytime(times[$n] - times[1]))

axso = Axis(fig[1, 1])
axηo = Axis(fig[1, 3])
axTo = Axis(fig[2, 1])
axeo = Axis(fig[2, 3])
axdye = Axis(fig[3, 1])
axSo = Axis(fig[3, 3])

hm = heatmap!(axso, soₙ, colorrange = (0, 0.25), colormap = :deep, nan_color = :lightgray)
Colorbar(fig[1, 2], hm, label = "Ocean Surface Speed (m s⁻¹)")

hm = heatmap!(axηo, ηoₙ, colorrange = (-.1, .5), colormap = :balance, nan_color = :lightgray)
Colorbar(fig[1, 4], hm, label = "Sea Surface Height (m)")

hm = heatmap!(axTo, Toₙ, colorrange = (26, 32), colormap = :magma, nan_color = :lightgray)
Colorbar(fig[2, 2], hm, label = "Surface Temperature (ᵒC)")

hm = heatmap!(axeo, eoₙ, colorrange = (0, 3e-4), colormap = :solar, nan_color = :lightgray)
Colorbar(fig[2, 4], hm, label = "Turbulent Kinetic Energy (m² s⁻²)")

hm = heatmap!(axdye, dyeₙ, colorrange = (-8, 0), colormap = :viridis, nan_color = :lightgray)
Colorbar(fig[3, 2], hm, label = "log₁₀(passive dye + 1e-16)")

hm = heatmap!(axSo, Soₙ, colorrange = (32, 37), colormap = :haline, nan_color = :lightgray)
Colorbar(fig[3, 4], hm, label = "Surface Salinity (g kg^-1)")

for ax in (axso, axηo, axTo, axeo, axdye, axSo)
    hidedecorations!(ax)
end

Label(fig[0, :], title)


# update img to be the last saved timestep
n[] = Nt

save("amazon_salinity_$(run_name)_snapshot.png", fig)

In [ ]:
CairoMakie.record(fig, "amazon_salinity_$(run_name).mp4", 1:Nt; framerate = 8) do nn
    n[] = nn
end

In [ ]:
using Oceananigans
using CairoMakie

# load the saved 3d dye output
dye_series = FieldTimeSeries(dye_3d_filename, "dye")

Nt = length(dye_series.times)

time_days = dye_series.times ./ days
minimum_dye = zeros(Float64, Nt)
maximum_dye = zeros(Float64, Nt)

for n in 1:Nt
    dye_snapshot = Array(interior(dye_series[n]))

    minimum_dye[n] = minimum(dye_snapshot)
    maximum_dye[n] = maximum(dye_snapshot)
end

# magnitude of the negative dye values
negative_dye_amount = max.(-minimum_dye, 0)

fig = Figure(size = (900, 700))

ax1 = Axis(
    fig[1, 1],
    xlabel = "Time (days)",
    ylabel = "Maximum dye concentration",
    title = "Maximum dye concentration throughout the run"
)

lines!(
    ax1,
    time_days,
    maximum_dye;
    linewidth = 3
)

scatter!(
    ax1,
    time_days,
    maximum_dye;
    markersize = 6
)

ax2 = Axis(
    fig[2, 1],
    xlabel = "Time (days)",
    ylabel = "Magnitude of negative dye",
    title = "Negative dye magnitude throughout the run",
    yscale = log10
)

# avoid plotting exact zeros on a logarithmic axis
positive_negative_indices = findall(negative_dye_amount .> 0)

lines!(
    ax2,
    time_days[positive_negative_indices],
    negative_dye_amount[positive_negative_indices];
    linewidth = 3
)

scatter!(
    ax2,
    time_days[positive_negative_indices],
    negative_dye_amount[positive_negative_indices];
    markersize = 6
)

save("amazon_$(output_tag)_dye_extrema.png", fig)

fig